### Import

In [91]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 100
LEVEL = "high"
SEED = 4

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, _, _, CRATE, DRATE = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)
INEFF_BATT = 0.95
# INEFF_INT = np.full(I, 0.99)
# INEFF_EXT = np.full(I, 0.98)
INEFF_INT = np.random.uniform(0.99, 0.999, I)
INEFF_EXT = np.random.uniform(0.95, 0.98, I)

✅ 총 10개 파일을 불러왔습니다: 1033.csv, 1818.csv, 2502.csv, 2503.csv, 2634.csv, 2698.csv, 2816.csv, 545.csv, 665.csv, 690.csv
📊 데이터 Shape: I=10, T=24, S=100
✅ 시뮬레이션 초기화 완료: S=100, Randomness='high', Random Seed=4, M1=3735.41, M2=10638.29
   - 개별 K 값: [200. 200. 400. 600. 100. 600. 300. 200. 300. 200.]


In [92]:
print(INEFF_INT)
print(INEFF_EXT)

[0.99365365 0.99533445 0.99587025 0.99432297 0.99002017 0.99369343
 0.99642526 0.99499062 0.9969028  0.99473499]
[0.97114219 0.97067914 0.97512628 0.97621828 0.97019284 0.96068472
 0.95092145 0.96711738 0.95840989 0.97465626]


### Individual Optimization (original)

In [93]:
m1 = gp.Model("individual")
m1.setParam("MIPGap", 1e-5)

x_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
z_ind = m1.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m1.update()

obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
m1.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m1.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x_ind[i, t] == (1/INEFF_EXT[i]) * yp_ind[i, t, s] - INEFF_EXT[i] * ym_ind[i, t, s] + zc_ind[i, t, s] - zd_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF_BATT <= z_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF_BATT <= DRATE[i])
    m1.addConstr(zc_ind[i, t, s]*INEFF_BATT <= K[i] - z_ind[i, t, s])
    m1.addConstr(zc_ind[i, t, s]*INEFF_BATT <= CRATE[i])
    m1.addConstr(z_ind[i, t, s] <= K[i])
    m1.addConstr(z_ind[i, t + 1, s] == z_ind[i, t, s] + INEFF_BATT * zc_ind[i, t, s] - zd_ind[i, t, s] / INEFF_BATT)

for i, s in product(range(I), range(S)): m1.addConstr(z_ind[i, 0, s] == K0[i])

m1.optimize()

if m1.status == GRB.OPTIMAL:
    x_ind = np.array([[x_ind[i, t].X for t in range(T)] for i in range(I)])
    yp_ind = np.array([[[yp_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_ind = np.array([[[ym_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_ind = np.array([[[zc_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_ind = np.array([[[zd_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_ind = np.array([[[z_ind[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; OBJ_IND = m1.objVal
    # phi1_ind = np.array([[[phi1_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_ind = np.array([[[phi2_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])

Set parameter MIPGap to value 1e-05
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-05

Optimize a model with 169000 rows, 121240 columns and 385000 nonzeros
Model fingerprint: 0xe6d8c95f
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 98923 rows and 27876 columns
Presolve time: 0.13s
Presolved: 70077 rows, 93364 columns, 322308 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.52s

Barrier statistics:
 AA' NZ     : 1.428e+06
 Factor NZ  : 7.639e+06 (roughly 130 MB of memory)
 Factor Ops : 2.371e+09 (less than 1 second per iteration)
 Threads    : 22

         

In [94]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 70)
print("\n[Individual]") ; print(header)
for t in range(7, 22):
    # R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_ind[:, t].sum()
    # yp_avg = np.mean([yp_ind[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_ind[:, t, s].sum() for s in range(S)])
    # zc_avg = np.mean([zc_ind[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_ind[:, t, s].sum() for s in range(S)]) 
    # z_avg = np.mean([z_ind[:, t, s].sum() for s in range(S)])

    i=1
    R_avg = np.mean([R[i, t, s] for s in range(S)]) ; x_sum = x_ind[i, t]
    yp_avg = np.mean([yp_ind[i, t, s] for s in range(S)]) ; ym_avg = np.mean([ym_ind[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_ind[i, t, s] for s in range(S)]) ; zd_avg = np.mean([zd_ind[i, t, s] for s in range(S)]) 
    z_avg = np.mean([z_ind[i, t, s] for s in range(S)])

    print(f"{t:>2} | {R_avg:>8.2f} {(1/INEFF_EXT[i]) * x_sum:>8.2f} {(1/INEFF_EXT[i]) * yp_avg:>8.2f} {INEFF_EXT[i] * ym_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[Individual]
 t |        R        x       y+       y-       zc       zd        z
----------------------------------------------------------------------
 7 |    75.98     0.54    32.62     0.00    42.97     0.15    24.20
 8 |   129.50   100.81    15.85     1.15    26.06    12.08    64.86
 9 |   157.00   113.11    32.88     2.52    27.36    13.83    76.91
10 |   189.17   151.90    30.57     8.29    28.45    13.46    88.35
11 |   226.01   168.90    53.49     8.38    27.79    15.78   101.21
12 |   228.29   174.86    52.47     8.53    24.20    14.71   110.99
13 |   515.87     0.00   556.72     0.00     1.51    42.36   118.50
14 |   467.18     0.00   465.03     0.00    20.15    17.99    75.35
15 |   474.47   460.44    68.17    59.67    26.33    20.80    75.55
16 |   160.22   147.04    15.88     8.64    23.83    17.89    78.67
17 |   191.61   173.48    22.19    14.14    25.58    15.50    82.48
18 |   245.26   195.51    63.86    13.76    18.26    18.60    90.46
19 |   113.65    97.76    20.21

### Holistic Optimization (Linear Decision Rule + MILP - M) (original)

In [104]:
m2 = gp.Model("holistic_MILP_M")
m2.setParam("MIPGap", 1e-5)
m2.setParam(GRB.Param.TimeLimit, 1200)

x_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x") ; yp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = m2.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z") ; zc_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m2.update()

obj_lin = gp.quicksum(
    P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)
) + gp.quicksum(
    (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
    for i in range(I) for t in range(T) for s in range(S)
)

eps = 1e-8
# quad_reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
obj = obj_lin - eps * quad_reg

# NOTE
m2.setObjective(obj, GRB.MAXIMIZE)
# m2.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m2.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x_hol[i, t] == (1 / INEFF_EXT[i]) * yp_hol[i, t, s] - INEFF_EXT[i] * ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF_BATT <= z_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF_BATT <= DRATE[i])
    m2.addConstr(zc_hol[i, t, s]*INEFF_BATT <= K[i] - z_hol[i, t, s])
    m2.addConstr(zc_hol[i, t, s]*INEFF_BATT <= CRATE[i])
    m2.addConstr(z_hol[i, t, s] <= K[i])
    m2.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + INEFF_BATT * zc_hol[i, t, s] - zd_hol[i, t, s] / INEFF_BATT)
for i, s in product(range(I), range(S)): m2.addConstr(z_hol[i, 0, s] == K0[i])

# NOTE : inefficiency 추가
balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = m2.addConstr(
        gp.quicksum(INEFF_INT[i] * dp_hol[i, t, s] for i in range(I))
        == gp.quicksum((1 / INEFF_INT[i]) * dm_hol[i, t, s] for i in range(I)),
        name=f"balance_{t}_{s}",
    )

m2.optimize()

if m2.status == GRB.OPTIMAL or m2.status == GRB.TIME_LIMIT:
    x_hol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_hol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_hol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_hol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_hol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_hol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_hol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_hol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = m2.objVal
    OBJ_HOL = sum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + sum(
        (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
        for i in range(I)
        for t in range(T)
        for s in range(S)
    )
    QUAD_HOL = eps * sum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

    lambda_dual = np.zeros((T, S))
    for t, s in product(range(T), range(S)): 
        lambda_dual[t, s] = balance_constraints[t, s].Pi
    print("Direct dual extraction successful!")

else:
    print(f"⚠️ Model finished with status: {m2.status}")

Set parameter MIPGap to value 1e-05
Set parameter TimeLimit to value 1200
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  1200
MIPGap  1e-05

Optimize a model with 171400 rows, 169240 columns and 481000 nonzeros
Model fingerprint: 0x8090ffda
Model has 48000 quadratic objective terms
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  QObjective range [2e-10, 2e-10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 96000 rows and 22020 columns
Presolve time: 0.14s
Presolved: 75400 rows, 147220 columns, 430000 nonzeros
Presolved model has 48000 quadratic objective terms
Ordering time: 1.54s
Ordering time: 1.56s

Barrier statistics:
 Dense cols : 220
 AA' NZ     : 4.450e+05
 Factor NZ  

In [105]:
header = (
    f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8} | {'BalErr':>8}\n"
    + "-" * 105
)
print(f"\n[HOLISTIC] Objective Value = {OBJ_HOL:.2f}, QUAD_HOL = {QUAD_HOL:>2f}")
print(header)

for t in range(0, 24):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    x_sum = np.sum(x_hol[:, t] * (1 / INEFF_EXT))
    yp_avg = np.mean([np.sum(yp_hol[:, t, s] * (1 / INEFF_EXT)) for s in range(S)])
    ym_avg = np.mean([np.sum(ym_hol[:, t, s] * INEFF_EXT) for s in range(S)])
    dp_avg = np.mean([np.sum(dp_hol[:, t, s] * INEFF_INT) for s in range(S)])
    dm_avg = np.mean([np.sum(dm_hol[:, t, s] * (1 / INEFF_INT)) for s in range(S)])
    bal_err = dp_avg - dm_avg

    print(
        f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f} | {bal_err:>8.4f}"
    )


[HOLISTIC] Objective Value = 5424823.89, QUAD_HOL = 0.379359
 t |        R        x       y+       y-       d+       d-       zc       zd        z |   BalErr
---------------------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |   0.0000
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |   0.0000
 2 |     0.00     0.00     0.00     1.08     0.00     0.00     1.08     0.00     0.00 |   0.0000
 3 |     0.00     0.00     0.00    57.17    10.84    10.84    57.07     0.00     1.03 |   0.0000
 4 |     0.00     0.00     0.00    92.86    19.46    19.46    92.70     0.00    55.25 |   0.0000
 5 |     0.00     0.00     0.00     5.31     0.00     0.00     5.31     0.00   143.31 |   0.0000
 6 |    54.55     0.00     4.32     0.00    24.63    24.63    49.98     0.00   148.36 |  -0.0000
 7 |   586.20     0.00    74.22     0.00   209.85   209.

In [106]:
for i, t, s in product(range(I), range(T), range(S)):
    if dp_hol[i, t, s] > 0.001 and dm_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, dm={dm_hol[i, t, s]}")
    if zc_hol[i, t, s] > 0.001 and zd_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, zc={zc_hol[i, t, s]}, zd={zd_hol[i, t, s]}")
    if dp_hol[i, t, s] > 0.001 and ym_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, ym={ym_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_hol[i, t, s] > 0.001 and yp_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dm={dm_hol[i, t, s]}, yp={yp_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

i=2, t=3, s=35, dp=53.430439961731885, ym=162.7415754966773, P_RT=46.367364260073934, P_IN=147.55031577626013, P_PN=143.286
i=2, t=3, s=38, dp=318.63586111541025, ym=434.71191475057833, P_RT=68.74232616710289, P_IN=147.55032124115323, P_PN=143.286
i=2, t=3, s=65, dp=318.63585962815426, ym=434.71191316447795, P_RT=65.6507442309124, P_IN=147.5503212411532, P_PN=143.286
i=2, t=3, s=86, dp=79.51952181240148, ym=189.49614459349408, P_RT=39.908680237688074, P_IN=147.5503163501966, P_PN=143.286
i=2, t=3, s=87, dp=318.6358610416271, ym=434.71191469692224, P_RT=60.61906058318571, P_IN=147.55032124115323, P_PN=143.286
i=2, t=4, s=19, dp=159.08012684234103, ym=271.08619836741906, P_RT=33.62693202403794, P_IN=146.85419966155519, P_PN=142.61
i=2, t=4, s=20, dp=149.21171918733637, ym=260.9660606488967, P_RT=61.87365116572502, P_IN=146.8541994593697, P_PN=142.61
i=2, t=4, s=35, dp=211.69702378027895, ym=325.0452573791892, P_RT=61.861068378195085, P_IN=146.85420073382758, P_PN=142.61
i=2, t=4, s=38, d

### Individual Replay

In [107]:
data = []

for t in range(T):
    s_fixed = 0
    data.append({
        'Time': t, 
        'P_DA': round(P_DA[t], 2), 
        'P_RT_avg': round(P_RT[t, s_fixed], 5), 
        'Lambda': round(-lambda_dual[t, s_fixed] * S, 5), 
        'P_PN_avg': round(P_PN[t, s_fixed], 4)
    })

pd.DataFrame(data)

# data_for_csv = []

# for t in range(T):
#     for s in range(S):
#         row = {
#             "t": t,
#             "s": s,
#             "P_DA": P_DA[t],
#             "P_RT": P_RT[t, s],
#             "P_PN": P_PN[t, s],
#             "P_IN": -lambda_dual[t, s] * S,
#         }
#         data_for_csv.append(row)

# df = pd.DataFrame(data_for_csv)

# output_filename = f"optimization_results_{SEED}.csv"
# df.to_csv(output_filename, index=False, encoding="utf-8-sig")

# print(f"✅ 데이터가 '{output_filename}' 파일로 성공적으로 저장되었습니다.")

,Time,P_DA,P_RT_avg,Lambda,P_PN_avg
0,0,91.140,84.127,178.287,182.286
1,1,80.280,35.884,153.196,160.550
2,2,74.560,32.992,138.933,149.110
3,3,71.640,61.969,128.394,143.286
4,4,71.310,37.963,118.049,142.610
5,5,74.540,76.746,107.643,153.492
6,6,80.820,49.292,73.729,161.642
7,7,87.240,37.423,73.729,174.486
8,8,101.970,44.550,43.262,203.944
9,9,110.370,43.271,42.199,220.740


In [108]:
m5 = gp.Model("DER_Individual_Replay")
# m5.setParam("MIPGap", 1e-5)

x = m5.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z = m5.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m5.update()

obj_lin = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(
        lambda_dual[t, s] * ((1/INEFF_INT[i]) * dm[i, t, s] - INEFF_INT[i] * dp[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    )
)

eps = 1e-8
# quad_reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp[i, t, s] * dp[i, t, s] + dm[i, t, s] * dm[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

obj = obj_lin - eps * quad_reg

m5.setObjective(obj, GRB.MAXIMIZE)
# m5.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m5.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x[i, t] == (1 / INEFF_EXT[i]) * yp[i, t, s] - INEFF_EXT[i] * ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
    m5.addConstr(zd[i, t, s]/INEFF_BATT <= z[i, t, s]) ; m5.addConstr(zc[i, t, s]*INEFF_BATT <= K[i] - z[i, t, s]) ; m5.addConstr(z[i, t, s] <= K[i])
    m5.addConstr(zd[i, t, s]/INEFF_BATT <= DRATE[i]) ; m5.addConstr(zc[i, t, s]*INEFF_BATT <= CRATE[i])
    m5.addConstr(z[i, t + 1, s] == z[i, t, s] + INEFF_BATT * zc[i, t, s] - zd[i, t, s] / INEFF_BATT)
for i, s in product(range(I), range(S)): m5.addConstr(z[i, 0, s] == K0[i])

m5.optimize()

if m5.status == GRB.OPTIMAL:
    print(f"Optimal solution found! Objective value: {m5.objVal}")
else:
    print("No optimal solution found.")


x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
zc_re = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_re = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])

OBJ_RE = (
    sum(P_DA[t] * x_re[i, t] for i in range(I) for t in range(T))
    + sum(
        (1 / S) * (P_RT[t, s] * yp_re[i, t, s] - P_PN[t, s] * ym_re[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    )
    + sum(
        lambda_dual[t, s] * (
            sum((1/INEFF_INT[i]) * dm_re[i, t, s] for i in range(I)) 
            - sum(INEFF_INT[i] * dp_re[i, t, s] for i in range(I))
        )
        for t in range(T) for s in range(S)
    )
)
QUAD_RE = eps * sum((1 / S) * (dp_re[i, t, s] * dp_re[i, t, s] + dm_re[i, t, s] * dm_re[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 169000 rows, 169240 columns and 433000 nonzeros
Model fingerprint: 0x10b0efcb
Model has 48000 quadratic objective terms
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  QObjective range [2e-10, 2e-10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 96000 rows and 22020 columns
Presolve time: 0.14s
Presolved: 73000 rows, 147220 columns, 382000 nonzeros
Presolved model has 48000 quadratic objective terms
Ordering time: 0.48s

Barrier statistics:
 AA' NZ     : 1.488e+06
 Factor NZ  : 7.850e+06 (roughly 150 MB of memory)
 Factor Ops : 2.383e+09 (less than 1 second per iteration)
 Threads    : 24

                  Objective                Residual


In [109]:
print(round((1/INEFF_EXT[i]) * x_ind[:,:].sum(),2), round((1/INEFF_EXT[i]) * x_re[:,:].sum(),2), round((1/INEFF_EXT[i]) * x_hol[:,:].sum(),2))

29212.68 33362.17 33383.22


In [110]:
for i, t, s in product(range(I), range(T), range(S)):
    if dp_re[i, t, s] > 0.001 and dm_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, dm={dm_re[i, t, s]}")
    if zc_re[i, t, s] > 0.001 and zd_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, zc={zc_re[i, t, s]}, zd={zd_re[i, t, s]}")
    if dp_re[i, t, s] > 0.001 and ym_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, ym={ym_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_re[i, t, s] > 0.001 and yp_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dm={dm_re[i, t, s]}, yp={yp_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

i=2, t=3, s=35, dp=44.94553213115197, ym=154.04024371152985, P_RT=46.367364260073934, P_IN=147.55031577626013, P_PN=143.286
i=2, t=3, s=38, dp=316.7604231762567, ym=432.7886449146913, P_RT=68.74232616710289, P_IN=147.55032124115323, P_PN=143.286
i=2, t=3, s=65, dp=316.76042234300064, ym=432.7886440579837, P_RT=65.6507442309124, P_IN=147.5503212411532, P_PN=143.286
i=2, t=3, s=86, dp=73.39745471103016, ym=183.21792382876404, P_RT=39.908680237688074, P_IN=147.5503163501966, P_PN=143.286
i=2, t=3, s=87, dp=316.76042388648744, ym=432.78864564383224, P_RT=60.61906058318571, P_IN=147.55032124115323, P_PN=143.286
i=2, t=4, s=19, dp=155.64683534856817, ym=267.565338564574, P_RT=33.62693202403794, P_IN=146.85419966155519, P_PN=142.61
i=2, t=4, s=20, dp=145.58692964696058, ym=257.2488225545588, P_RT=61.87365116572502, P_IN=146.8541994593697, P_PN=142.61
i=2, t=4, s=35, dp=209.00981276714205, ym=322.289509742874, P_RT=61.861068378195085, P_IN=146.85420073382758, P_PN=142.61
i=2, t=4, s=38, dp=316

In [111]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8} | {'BalErr':>8}\n" + "-" * 105)
print(f"\n[REPLAY] Objective Value = {OBJ_RE:.2f}, QUAD_RE = {QUAD_RE:>2f}") 
print(header)

for t in range(T):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    x_sum = np.sum(x_re[:, t] * (1 / INEFF_EXT))
    yp_avg = np.mean([np.sum(yp_re[:, t, s] * (1 / INEFF_EXT)) for s in range(S)])
    ym_avg = np.mean([np.sum(ym_re[:, t, s] * INEFF_EXT) for s in range(S)])
    dp_avg = np.mean([np.sum(dp_re[:, t, s] * INEFF_INT) for s in range(S)])
    dm_avg = np.mean([np.sum(dm_re[:, t, s] * (1 / INEFF_INT)) for s in range(S)])
    bal_err = dp_avg - dm_avg

    print(
        f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f} | {bal_err:>8.4f}"
    )


[REPLAY] Objective Value = 5424823.90, QUAD_RE = 0.379466
 t |        R        x       y+       y-       d+       d-       zc       zd        z |   BalErr
---------------------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |  -0.0000
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |  -0.0000
 2 |     0.00     0.00     0.00     1.01     0.00     0.00     1.01     0.00     0.00 |  -0.0000
 3 |     0.00     0.00     0.00    56.45    10.64    10.88    56.59     0.00     0.96 |  -0.2353
 4 |     0.00     0.00     0.00    91.51    19.25    19.27    91.37     0.00    54.72 |  -0.0222
 5 |     0.00     0.00     0.00     5.67     0.00     0.00     5.67     0.00   141.52 |   0.0000
 6 |    54.55     0.00     3.39     0.00    28.99     5.14    27.14     0.00   146.90 |  23.8457
 7 |   586.20     0.00    73.42     0.00   209.99   213.13 

In [112]:
# Aggregator(VPP)가 외부망과 거래할 때 적용할 대표 효율 (예: 평균 효율)
INEFF_AGG_EXT = np.mean(INEFF_EXT)

print("=" * 50)
print("AGGREGATOR LOSS ANALYSIS")
print("=" * 50)
total_losses = []
for t in range(T):
    scenario_losses = []
    for s in range(S):
        # 1. 내부 시장 (Internal Pool) 수급 불균형 (개별 효율 [i] 적용)
        # numpy array끼리의 곱셈은 element-wise로 자동 처리됨
        # sum( dp[i] * ineff[i] )
        total_supply = np.sum(dp_re[:, t, s] * INEFF_INT)
        # sum( dm[i] * (1/ineff[i]) )
        total_demand = np.sum(dm_re[:, t, s] * (1 / INEFF_INT))

        # 내부 시장에서 걷은 돈의 단가 (Lambda)
        lambda_price = -lambda_dual[t, s] * S

        # 수급 불균형량
        imbalance = total_demand - total_supply

        # [수정 포인트] VPP 대표 외부 효율(INEFF_AGG_EXT)을 고려한 유효 P_RT 계산
        # Aggregator가 부족분을 사오거나 팔 때는 VPP 전체의 대표 효율(평균 등)을 적용
        if imbalance > 0:
            # 부족함 -> 외부에서 사와야 함 (Buy)
            effective_p_rt = P_RT[t, s] * (1 / INEFF_AGG_EXT)
        else:
            # 남음 -> 외부로 팔아야 함 (Sell)
            effective_p_rt = P_RT[t, s] * INEFF_AGG_EXT

        # 손실(재정적 잉여) 계산
        loss = imbalance * (lambda_price - effective_p_rt)

        scenario_losses.append(loss)

    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

# [수정 1] 외부 시장 손실 분석 (External Grid Loss) - 개별 효율 [i] 반영
print("=" * 50)
print("EXTERNAL GRID LOSS ANALYSIS")
print("=" * 50)
total_ext_loss_energy = 0
for i, t, s in product(range(I), range(T), range(S)):
    # 1. Real-Time 판매(yp) 손실: yp * (1/eta_i - 1)
    loss_export_rt = yp_re[i, t, s] * (1 / INEFF_EXT[i] - 1)

    # 2. Real-Time 구매(ym) 손실: ym * (1 - eta_i)
    loss_import_rt = ym_re[i, t, s] * (1 - INEFF_EXT[i])

    # 3. Day-Ahead 판매(x) 손실: x * (1/eta_i - 1)
    loss_export_da = x_re[i, t] * (1 / INEFF_EXT[i] - 1)

    total_ext_loss_energy += loss_export_rt + loss_import_rt + loss_export_da

print(f"Total Physical Energy Lost in External Grid: {total_ext_loss_energy:.2f} kWh")


print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Internal Aggregator Loss (Financial): {total_loss:.2f}")
print("Realized Profit (Replay + AggLoss)", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)

print()
print("=" * 60)
print("INDIVIDUAL PROFIT ANALYSIS")
print("=" * 60)
profit_ind = np.zeros(I)
profit_re = np.zeros(I)
profit_hol = np.zeros(I)
profit_re_adjusted = np.zeros(I)

price_weighted_usage = np.zeros(I)
for i, t, s in product(range(I), range(T), range(S)):
    lambda_price = -lambda_dual[t, s] * S

    # [내부 시장] 가중치 계산 (개별 효율 [i] 적용)
    dm_contribution = dm_re[i, t, s] * (lambda_price * (1 / INEFF_INT[i]))
    dp_contribution = dp_re[i, t, s] * (lambda_price * INEFF_INT[i])

    price_weighted_usage[i] += dm_contribution + dp_contribution

total_price_weighted_usage = np.sum(price_weighted_usage)
print(f"Total price-weighted usage: {total_price_weighted_usage:.2f}")

for i in range(I):
    # 1. Individual Case
    profit_ind[i] = 0
    for t in range(T):
        profit_ind[i] += P_DA[t] * x_ind[i, t]
        profit_ind[i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
        profit_ind[i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])

    # 2. Replay Case
    profit_re[i] = 0
    for t in range(T):
        profit_re[i] += P_DA[t] * x_re[i, t]

        profit_re[i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])

        lambda_price = -lambda_dual[t, :] * S

        # [내부 시장] 수익/비용 계산 (개별 효율 [i] 적용)
        profit_re[i] += np.mean(
            [lambda_price[s] * dp_re[i, t, s] * INEFF_INT[i] for s in range(S)]
        )
        profit_re[i] -= np.mean(
            [lambda_price[s] * dm_re[i, t, s] * (1 / INEFF_INT[i]) for s in range(S)]
        )

    # 3. Holistic Case
    profit_hol[i] = 0
    for t in range(T):
        profit_hol[i] += P_DA[t] * x_hol[i, t]
        profit_hol[i] += np.mean([P_RT[t, s] * yp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([P_PN[t, s] * ym_hol[i, t, s] for s in range(S)])

        lambda_price = -lambda_dual[t, :] * S

        # [내부 시장] Holistic 비교 시 (개별 효율 [i] 적용)
        profit_hol[i] += np.mean(
            [lambda_price[s] * dp_hol[i, t, s] * INEFF_INT[i] for s in range(S)]
        )
        profit_hol[i] -= np.mean(
            [lambda_price[s] * dm_hol[i, t, s] * (1 / INEFF_INT[i]) for s in range(S)]
        )

# 손실 배분 및 최종 이익 계산
loss_per_player = np.zeros(I)
for i in range(I):
    if total_price_weighted_usage > 0:
        loss_per_player[i] = total_loss * (
            price_weighted_usage[i] / total_price_weighted_usage
        )
    else:
        loss_per_player[i] = total_loss / I
    profit_re_adjusted[i] = profit_re[i] + loss_per_player[i]

# 헤더 출력
print(
    f"{'Player':<8} {'Individual':<12} {'Replay':<12} {'Re+Loss':<12} {'Holistic':<12} {'Alloc Loss':<12} {'Gain ($)':<12} {'Gain (%)':<22}"
)
print("-" * 125)

total_ind = 0
total_re = 0
total_hol = 0
total_re_adj = 0

for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]

    if profit_ind[i] != 0:
        percentage_change = (diff_adj_ind / profit_ind[i]) * 100
        final_percentage_str = (
            f"(+{percentage_change:.1f}%)"
            if percentage_change >= 0
            else f"({percentage_change:.1f}%)"
        )
    else:
        final_percentage_str = "(N/A)"

    print(
        f"{i:<8} {profit_ind[i]:<12.2f} {profit_re[i]:<12.2f} {profit_re_adjusted[i]:<12.2f} {profit_hol[i]:<12.2f} {loss_per_player[i]:<12.2f} {diff_adj_ind:<12.2f} {final_percentage_str:<22}"
    )

    total_ind += profit_ind[i]
    total_re += profit_re[i]
    total_hol += profit_hol[i]
    total_re_adj += profit_re_adjusted[i]

total_diff_adj_ind = total_re_adj - total_ind
if total_ind != 0:
    total_percentage_change = (total_diff_adj_ind / total_ind) * 100
    total_final_percentage_str = (
        f"(+{total_percentage_change:.1f}%)"
        if total_percentage_change >= 0
        else f"({total_percentage_change:.1f}%)"
    )
else:
    total_final_percentage_str = "(N/A)"

print("-" * 125)
print(
    f"{'TOTAL':<8} {total_ind:<12.2f} {total_re:<12.2f} {total_re_adj:<12.2f} {total_hol:<12.2f} {np.sum(loss_per_player):<12.2f} {total_diff_adj_ind:<12.2f} {total_final_percentage_str:<22}"
)

AGGREGATOR LOSS ANALYSIS
EXTERNAL GRID LOSS ANALYSIS
Total Physical Energy Lost in External Grid: 112341.05 kWh

SUMMARY
Individual Participation Profit 4668826.171107661
Expected Replay Profit 5424823.897897502
Internal Aggregator Loss (Financial): -2279.74
Realized Profit (Replay + AggLoss) 5422544.15653409
Holistic Profit 5424823.888389004

INDIVIDUAL PROFIT ANALYSIS
Total price-weighted usage: 469510917.52
Player   Individual   Replay       Re+Loss      Holistic     Alloc Loss   Gain ($)     Gain (%)              
-----------------------------------------------------------------------------------------------------------------------------
0        325232.62    372237.28    372192.18    372237.28    -45.09       46959.56     (+14.4%)              
1        383457.64    432560.83    432520.71    432560.83    -40.12       49063.07     (+12.8%)              
2        589307.13    676605.88    675804.58    676605.88    -801.30      86497.44     (+14.7%)              
3        827263.19  